# P1 — Rule-Based Strategies (5m perp, Q1+Q2 2024)

**Slug**: `crypto_intraday/11_rule_based`

**Status**: in_progress

**Research question**: Among the canonical rule-based strategy families — mean-reversion (RSI, Bollinger %B, distance-from-MA), breakout (Bollinger, Donchian), trend (MA crossover, MACD), and volume-shock — which (if any) survive realistic costs on Binance BTCUSDT / ETHUSDT perp at 5m sampling?

**Scope** (driven by P1 horizon-discovery decision):
- Interval: 5m.
- Window: Q1+Q2 2024 (same train slice as the horizon discovery notebook).
- Focus on mean-reversion (rank-IC said so); trend / breakout included as explicit null comparisons.
- Per-symbol slippage from `configs/crypto_intraday.yaml`.
- Funding included in `perp_base` and `perp_stress` scenarios.

**Decision rule**: a strategy is `accept_monitoring` only if its net return (after `perp_stress` costs) is positive AND its Sharpe is > 0.5 AND turnover is feasible (annualized < ~1000× — extreme but plausible at 5m). Anything that needs `zero` or `perp_base` to look good is `needs_revision` or `reject`.

In [ ]:
from dataclasses import dataclass

import numpy as np
import pandas as pd

from alpha_lab.backtest.metrics import summary
from alpha_lab.backtest.vector import run_backtest
from alpha_lab.data.holdout import PMHoldout, access_summary_for_report
from alpha_lab.data.loaders import binance_vision as bv
from alpha_lab.features import intraday as f
from alpha_lab.utils.config import load_config

pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.width', 200)

In [ ]:
cfg = load_config('crypto_intraday')
holdout = PMHoldout.from_config('crypto_intraday')

SYMBOLS = ['BTCUSDT', 'ETHUSDT']
MARKET = 'perp'
ARCHIVE_INTERVAL = '1m'
WORK_INTERVAL = '5min'
BARS_PER_YEAR_5M = 365 * 24 * 12  # 105,120
RES_START, RES_END = '2024-01-01', '2024-07-01'

In [ ]:
# Load 1m and resample (right-labeled)
panel_1m = bv.load_klines(
    SYMBOLS, ARCHIVE_INTERVAL, start=RES_START, end=RES_END,
    market=MARKET, holdout=holdout,
)

def resample_ohlcv(df: pd.DataFrame, offset: str) -> pd.DataFrame:
    agg = {'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last',
           'volume': 'sum', 'quote_volume': 'sum', 'n_trades': 'sum',
           'taker_buy_base': 'sum'}
    return df.resample(offset, label='right', closed='right').agg(agg).dropna(how='all')

ohlcv_5m = {sym: resample_ohlcv(df, WORK_INTERVAL) for sym, df in panel_1m.frames.items()}
prices_5m = pd.DataFrame({s: o['close'] for s, o in ohlcv_5m.items()})
common_idx = prices_5m.dropna().index
prices_5m = prices_5m.loc[common_idx]
ohlcv_5m = {s: o.loc[common_idx] for s, o in ohlcv_5m.items()}
print(f'5m bars: {len(common_idx):,}  [{common_idx.min()} → {common_idx.max()}]')

funding = bv.load_funding(SYMBOLS, start=RES_START, end=RES_END, holdout=holdout)
print(f'funding events: {len(funding)}')

## Strategy library

Each strategy is a function that returns a target-weight DataFrame indexed by `common_idx` with columns = `SYMBOLS`. Per-symbol weights are bounded so the gross is ≤ 1.0 (sum of absolute weights).

In [ ]:
PER_SYMBOL_CAP = 0.5  # so 2-symbol gross ≤ 1.0

def _zsig(s: pd.Series, window: int = 240) -> pd.Series:
    """Standardize a stationary signal by rolling std (min 60 obs)."""
    return s / s.rolling(window, min_periods=60).std()

def rsi_mr() -> pd.DataFrame:
    w = {}
    for sym in SYMBOLS:
        rsi = f.rsi(ohlcv_5m[sym]['close'], window=14)
        # 50-centered, scaled to roughly [-1, 1] for RSI in [30, 70]
        w[sym] = ((50 - rsi) / 20).clip(-1.0, 1.0) * PER_SYMBOL_CAP
    return pd.DataFrame(w).fillna(0.0)

def bollinger_mr() -> pd.DataFrame:
    w = {}
    for sym in SYMBOLS:
        bb = f.bollinger_pct_b(ohlcv_5m[sym]['close'], window=20)
        # %B < 0.5 → long, > 0.5 → short
        w[sym] = ((0.5 - bb) * 2).clip(-1.0, 1.0) * PER_SYMBOL_CAP
    return pd.DataFrame(w).fillna(0.0)

def bollinger_breakout() -> pd.DataFrame:
    w = {}
    for sym in SYMBOLS:
        bb = f.bollinger_pct_b(ohlcv_5m[sym]['close'], window=20)
        sig = pd.Series(0.0, index=bb.index)
        sig[bb > 0.95] = 1.0
        sig[bb < 0.05] = -1.0
        w[sym] = sig * PER_SYMBOL_CAP
    return pd.DataFrame(w).fillna(0.0)

def distma_reversion() -> pd.DataFrame:
    w = {}
    for sym in SYMBOLS:
        dma = f.distance_from_ma(ohlcv_5m[sym]['close'], window=60)
        z = _zsig(-dma)
        w[sym] = z.clip(-1.0, 1.0) * PER_SYMBOL_CAP
    return pd.DataFrame(w).fillna(0.0)

def short_term_reversal() -> pd.DataFrame:
    w = {}
    for sym in SYMBOLS:
        r = f.log_return(ohlcv_5m[sym]['close'], window=6)  # last 30m
        z = _zsig(-r)
        w[sym] = z.clip(-1.0, 1.0) * PER_SYMBOL_CAP
    return pd.DataFrame(w).fillna(0.0)

def ma_crossover() -> pd.DataFrame:
    w = {}
    for sym in SYMBOLS:
        fast = ohlcv_5m[sym]['close'].rolling(12).mean()
        slow = ohlcv_5m[sym]['close'].rolling(48).mean()
        sig = pd.Series(np.where(fast > slow, 1.0, -1.0), index=fast.index)
        w[sym] = sig * PER_SYMBOL_CAP
    return pd.DataFrame(w).fillna(0.0)

def macd_trend() -> pd.DataFrame:
    w = {}
    for sym in SYMBOLS:
        m = f.macd(ohlcv_5m[sym]['close'])
        w[sym] = pd.Series(np.where(m['hist'] > 0, 1.0, -1.0), index=m.index) * PER_SYMBOL_CAP
    return pd.DataFrame(w).fillna(0.0)

def donchian_breakout() -> pd.DataFrame:
    w = {}
    for sym in SYMBOLS:
        o = ohlcv_5m[sym]
        dp = f.donchian_position(o['high'], o['low'], o['close'], window=20)
        sig = pd.Series(0.0, index=dp.index)
        sig[dp > 0.95] = 1.0
        sig[dp < 0.05] = -1.0
        w[sym] = sig * PER_SYMBOL_CAP
    return pd.DataFrame(w).fillna(0.0)

def volume_shock_continuation() -> pd.DataFrame:
    w = {}
    for sym in SYMBOLS:
        o = ohlcv_5m[sym]
        vz = f.volume_zscore(o['volume'], window=60)
        r = f.log_return(o['close'], window=6)
        sig = pd.Series(0.0, index=vz.index)
        shock = vz > 2.0
        sig[shock & (r > 0)] = 1.0
        sig[shock & (r < 0)] = -1.0
        w[sym] = sig * PER_SYMBOL_CAP
    return pd.DataFrame(w).fillna(0.0)

STRATEGIES: dict[str, callable] = {
    'rsi_mr':                  rsi_mr,
    'bollinger_mr':            bollinger_mr,
    'bollinger_breakout':      bollinger_breakout,
    'distma_reversion':        distma_reversion,
    'short_term_reversal':     short_term_reversal,
    'ma_crossover':            ma_crossover,
    'macd_trend':              macd_trend,
    'donchian_breakout':       donchian_breakout,
    'volume_shock_continuation': volume_shock_continuation,
}
print('Strategies:', list(STRATEGIES.keys()))

## Cost scenarios (per-symbol slippage from config)

In [ ]:
@dataclass(frozen=True)
class CostScenario:
    name: str
    fee_bps: float
    slip_bps: dict[str, float]
    use_funding: bool

scenarios = []
for key in ['zero', 'perp_base', 'perp_stress']:
    block = cfg['costs'][key]
    scenarios.append(CostScenario(
        name=key,
        fee_bps=float(block['fee_bps_per_side']),
        slip_bps={s: float(block['slippage_bps_per_side'][s]) for s in SYMBOLS},
        use_funding=bool(block.get('include_funding', False)),
    ))
for s in scenarios:
    print(s)

## Run every (strategy, scenario)

In [ ]:
rows = []
raw_results: dict[tuple[str, str], object] = {}
for strat_name, strat_fn in STRATEGIES.items():
    weights = strat_fn().reindex(common_idx).fillna(0.0)
    for sc in scenarios:
        res = run_backtest(
            weights, prices_5m,
            costs_bps=sc.fee_bps,
            slippage_bps=sc.slip_bps,
            funding=funding if sc.use_funding else None,
            bars_per_year=BARS_PER_YEAR_5M,
        )
        raw_results[(strat_name, sc.name)] = res
        stats = summary(res.returns, periods=BARS_PER_YEAR_5M)
        rows.append({
            'strategy': strat_name,
            'scenario': sc.name,
            'gross_total': float((1 + res.gross_returns).prod() - 1),
            'net_total':   float((1 + res.returns).prod() - 1),
            'cost_drag':   float(res.costs.sum()),
            'funding_drag': float(res.funding_costs.sum()),
            'turnover_sum': float(res.turnover.sum()),
            'Sharpe':      stats.get('Sharpe', float('nan')),
            'MaxDD':       stats.get('MaxDD',  float('nan')),
        })
results = pd.DataFrame(rows)
results

## Net total return by (strategy, scenario)

In [ ]:
net_pivot = results.pivot(index='strategy', columns='scenario', values='net_total')
net_pivot = net_pivot[['zero', 'perp_base', 'perp_stress']]
net_pivot = net_pivot.reindex(index=list(STRATEGIES.keys()))
print('Net total return — Q1+Q2 2024:')
print(net_pivot)

In [ ]:
sharpe_pivot = results.pivot(index='strategy', columns='scenario', values='Sharpe')
sharpe_pivot = sharpe_pivot[['zero', 'perp_base', 'perp_stress']]
sharpe_pivot = sharpe_pivot.reindex(index=list(STRATEGIES.keys()))
print('Annualized Sharpe — Q1+Q2 2024:')
print(sharpe_pivot)

In [ ]:
# Cost / funding drag attribution at perp_base
attrib = results[results.scenario == 'perp_base'][
    ['strategy', 'gross_total', 'cost_drag', 'funding_drag', 'net_total', 'turnover_sum']
].set_index('strategy')
print('Attribution under perp_base costs:')
print(attrib)

## Survival decisions per strategy

In [ ]:
decisions = []
for strat in STRATEGIES:
    stress_row = results[(results.strategy == strat) & (results.scenario == 'perp_stress')].iloc[0]
    base_row = results[(results.strategy == strat) & (results.scenario == 'perp_base')].iloc[0]
    zero_row = results[(results.strategy == strat) & (results.scenario == 'zero')].iloc[0]
    net_stress = float(stress_row['net_total'])
    sharpe_stress = float(stress_row['Sharpe'])
    if net_stress > 0 and sharpe_stress > 0.5:
        decision = 'accept_monitoring'
    elif float(base_row['net_total']) > 0 and float(base_row['Sharpe']) > 0.0:
        decision = 'needs_revision'
    elif float(zero_row['net_total']) > 0 and float(zero_row['Sharpe']) > 0.0:
        decision = 'reject_cost_killed'
    else:
        decision = 'reject'
    decisions.append({
        'strategy': strat,
        'net_stress': net_stress,
        'sharpe_stress': sharpe_stress,
        'net_base':   float(base_row['net_total']),
        'net_zero':   float(zero_row['net_total']),
        'decision':   decision,
    })
decision_df = pd.DataFrame(decisions)
print(decision_df.to_string(index=False))

## PM holdout audit

In [ ]:
audit = access_summary_for_report()
print('PM Holdout audit:')
for k, v in audit.items():
    print(f'  {k}: {v}')
if audit['accessed']:
    raise RuntimeError('PM HOLDOUT WAS ACCESSED — contaminated.')
print('\nPM Holdout was not accessed.')

## Decision

Strategy-level decisions are recorded in `docs/research_decisions/crypto_intraday/P1-rule-based.md` once the notebook output is captured. The headline reading is:
- Net total return AT STRESS costs — the headline metric for tradability.
- Sharpe at stress — the noise-floor check.
- `reject_cost_killed` flag: strategy has positive gross / zero-cost return but loses under realistic costs. Common at 5m frequency.

Next: P2 (perp-specific + cross-asset + vol regimes).